In [1]:
"""
Physics Constants Relationship Analyzer - Streamlined Version
============================================================

A tool for discovering mathematical relationships between numerical coefficients
in fundamental physics equations.

This is a streamlined version with minimal enhancements to the original code.
Key improvements:
1. Better code organization and comments
2. Enhanced ratio scanning
3. Improved reporting
4. Simple visualization option

Author: Manus AI (based on original code)
Date: May 22, 2025
"""

import sympy as sp
import numpy as np
import math
import itertools
import time
from fractions import Fraction
from collections import defaultdict

# ============================================================
# 1.  Symbols & "unit" symbols to strip
# ============================================================
ℏ, c, G, k_B, α = sp.symbols('hbar c G k_B alpha', positive=True)
ε0, μ0, e, σ_SB, h = sp.symbols('epsilon_0 mu_0 e sigma_SB h', positive=True)
STRIP = {c, ℏ, G, k_B, h, ε0, μ0}

# ============================================================
# 2.  Equation library
# ============================================================
E, m, F, a, q1, q2, r, ν, T = sp.symbols('E m F a q1 q2 r nu T')
λ, ω, η, ρ, Δ = sp.symbols('lambda omega eta rho Delta')
Z0 = sp.symbols('Z_0')

# Dictionary of physics equations with their names as keys
EQU = {
    "Einstein E=mc²":        sp.Eq(E, m*c**2),
    "Newton II":             sp.Eq(F, m*a),
    "Planck E=hν":           sp.Eq(E, h*ν),
    "de Broglie λ":          sp.Eq(λ, ℏ/(m*c)),
    "Compton λ_c":           sp.Eq(λ, ℏ/(m*c)),
    "Coulomb law":           sp.Eq(F, 1/(4*sp.pi*ε0)*q1*q2/r**2),
    "Vacuum impedance":      sp.Eq(Z0, μ0*c),
    "Fine-structure α":      sp.Eq(α, e**2/(4*sp.pi*ε0*ℏ*c)),
    "Bohr radius":           sp.Eq(sp.Symbol('a0'), 4*sp.pi*ε0*ℏ**2/(m*e**2)),
    "Rydberg R∞":            sp.Eq(sp.Symbol('R_inf'), α**2*m*c/(2*h)),
    "Boltzmann kT":          sp.Eq(E, k_B*T),
    "Stefan–Boltzmann":      sp.Eq(σ_SB, sp.pi**2*k_B**4/(60*ℏ**3*c**2)),
    "Black-body u":          sp.Eq(sp.Symbol('u'), 8*sp.pi**5*k_B**4/(15*h**3*c**3)),
    "Casimir P":             sp.Eq(sp.Symbol('P_cas'), -sp.pi**2*ℏ*c/(240*a**4)),
    "BCS gap":               sp.Eq(Δ, 3.528*k_B*T),
    "ζ(3)":                  sp.Eq(sp.Symbol('zeta3'), sp.zeta(3)),
    "Catalan G":             sp.Eq(sp.Symbol('G_cat'), sp.Catalan),
    "Golden φ":              sp.Eq(sp.Symbol('phi'), (1+sp.sqrt(5))/2),
    # Additional equations
    "Gravitational force":   sp.Eq(F, G*m*sp.Symbol('m2')/r**2),
    "Schwarzschild radius":  sp.Eq(sp.Symbol('r_s'), 2*G*m/c**2),
}

# ============================================================
# 3.  Coefficient extraction functions
# ============================================================
def strip_units(expr):
    """Remove physical units (constants in STRIP) from expressions."""
    return sp.factor(expr.xreplace({u:1 for u in STRIP}), sp.pi, sp.E)

def gather_coeffs(eq, label):
    """
    Extract numerical coefficients from an equation.

    Args:
        eq: SymPy equation
        label: Name of the equation

    Returns:
        List of tuples (numerical value, string representation, equation label)
    """
    lst = []
    for side in (eq.lhs, eq.rhs):
        s = strip_units(side)
        for term in sp.Add.make_args(s.expand()):
            coeff, _ = term.as_coeff_Mul()
            if coeff not in (1, -1):
                lst.append((float(coeff.evalf(50)), str(coeff), label))
    return lst

# Extract coefficients from all equations
def extract_all_coeffs():
    """Extract coefficients from all equations in the library."""
    all_coeffs = []
    for name, eq in EQU.items():
        all_coeffs += gather_coeffs(eq, name)
    return all_coeffs

# ============================================================
# 4.  Constant catalogue
# ============================================================
def build_constant_catalogue():
    """Build a dictionary of mathematical constants for comparison."""
    TARGETS = {}

    # Basic constants
    TARGETS["π"] = np.pi
    TARGETS["e"] = np.e
    TARGETS["π²"] = np.pi**2
    TARGETS["√2"] = np.sqrt(2)
    TARGETS["√3"] = np.sqrt(3)

    # Multiples of π
    for n in range(1, 11):
        TARGETS[f"{n}π"] = n*np.pi

    # Fractions of π
    for n in range(1, 11):
        TARGETS[f"π/{n}"] = np.pi/n

    # Reciprocals
    TARGETS.update({f"1/({k})": 1/v for k, v in list(TARGETS.items())})

    return TARGETS

# ============================================================
# 5.  Ratio scanning
# ============================================================
def scan_ratios(all_coeffs, TARGETS, tolerance=1e-6):
    """
    Scan for ratio relationships between coefficients.

    Args:
        all_coeffs: List of coefficient tuples
        TARGETS: Dictionary of mathematical constants
        tolerance: Matching tolerance

    Returns:
        Dictionary of ratio hits
    """
    hits = defaultdict(list)

    for (v1, s1, e1), (v2, s2, e2) in itertools.combinations(all_coeffs, 2):
        if abs(v2) < 1e-50:  # Avoid division by zero or very small numbers
            continue

        r = v1 / v2

        # Check against known constants
        for lbl, val in TARGETS.items():
            # Check both direct and inverse relationships
            if (abs(r - val) < tolerance * abs(val) or
                abs(r - 1/val) < tolerance * abs(val)):
                relation = "≈" if abs(r - val) < tolerance * abs(val) else "≈ 1/"
                hits[(e1, e2)].append(f"{s1}/{s2} {relation} {lbl}")

        # Check for simple rational approximations
        q = Fraction(r).limit_denominator(50)
        if abs(r - float(q)) < tolerance * abs(float(q)):
            if max(abs(q.numerator), q.denominator) > 1:  # Skip trivial fractions
                hits[(e1, e2)].append(f"{s1}/{s2} ≈ {q.numerator}/{q.denominator}")

    return hits

# ============================================================
# 6.  Integer relation search
# ============================================================
def find_integer_relations(all_coeffs, max_terms=6, max_coeff=50, tolerance=1e-10):
    """
    Find integer relations between coefficients.

    Args:
        all_coeffs: List of coefficient tuples
        max_terms: Maximum number of terms in relations
        max_coeff: Maximum coefficient magnitude
        tolerance: Matching tolerance

    Returns:
        List of integer relation strings
    """
    values = [v for v, _, _ in all_coeffs]
    labels = [f"{s} ({e})" for _, s, e in all_coeffs]

    # Find relations
    relation = brute_pslq(values, max_terms, max_coeff, tolerance)

    if relation:
        # Format the relation for display
        terms = []
        for i, coef in enumerate(relation):
            if coef != 0:
                sign = "+" if coef > 0 and i > 0 else ""
                terms.append(f"{sign}{coef}·{labels[i]}")

        return [" ".join(terms) + " = 0"]

    return []

def brute_pslq(vals, max_terms=6, max_coeff=50, tol=1e-10):
    """
    Brute-force search for integer relations.

    Args:
        vals: List of values to check
        max_terms: Maximum number of terms in relations
        max_coeff: Maximum coefficient magnitude
        tol: Tolerance for zero

    Returns:
        List of coefficients or None
    """
    n = len(vals)

    # Try different subset sizes
    for k in range(2, min(max_terms, n) + 1):
        # Generate combinations of indices
        for idx in itertools.combinations(range(n), k):
            subset = [vals[i] for i in idx]

            # Skip if any value is too small
            if any(abs(v) < 1e-50 for v in subset):
                continue

            # Try coefficient combinations
            for coeffs in itertools.product(range(-max_coeff, max_coeff + 1), repeat=k):
                # Skip all zeros
                if all(c == 0 for c in coeffs):
                    continue

                # Skip if not coprime
                if math.gcd(*[abs(c) for c in coeffs if c != 0]) != 1:
                    continue

                # Check if linear combination is close to zero
                if abs(sum(c * v for c, v in zip(coeffs, subset))) < tol:
                    # Construct full relation vector
                    rel = [0] * n
                    for i, c in zip(idx, coeffs):
                        rel[i] = c
                    return rel

    return None

# ============================================================
# 7.  Simple visualization (optional)
# ============================================================
def create_simple_visualization(ratio_hits, filename="relationships.txt"):
    """
    Create a simple text visualization of the relationships.

    Args:
        ratio_hits: Dictionary of ratio hits
        filename: Output filename
    """
    with open(filename, 'w') as f:
        f.write("RELATIONSHIP NETWORK\n")
        f.write("===================\n\n")

        # Count relationships between equations
        equation_connections = defaultdict(int)
        for (e1, e2), hits in ratio_hits.items():
            equation_connections[e1] += len(hits)
            equation_connections[e2] += len(hits)

        # Sort equations by number of connections
        sorted_equations = sorted(equation_connections.items(),
                                 key=lambda x: x[1], reverse=True)

        f.write("Equations by number of relationships:\n")
        f.write("-----------------------------------\n")
        for eq, count in sorted_equations:
            f.write(f"{eq}: {count} relationships\n")

        f.write("\nDetailed relationships:\n")
        f.write("---------------------\n")
        for (e1, e2), hits in sorted(ratio_hits.items()):
            f.write(f"{e1} <-> {e2}:\n")
            for hit in hits:
                f.write(f"  {hit}\n")
            f.write("\n")

# ============================================================
# 8.  Main function
# ============================================================
def analyze_physics_relationships(tolerance=1e-6, max_terms=6, max_coeff=50,
                                 create_viz=True):
    """
    Run the complete analysis pipeline.

    Args:
        tolerance: Tolerance for ratio matching
        max_terms: Maximum terms in integer relations
        max_coeff: Maximum coefficient magnitude
        create_viz: Whether to create visualization

    Returns:
        Tuple of (ratio_hits, integer_relations)
    """
    print("\n" + "="*80)
    print("PHYSICS CONSTANTS RELATIONSHIP ANALYSIS")
    print("="*80 + "\n")

    # Extract coefficients
    print("Extracting coefficients from equations...")
    all_coeffs = extract_all_coeffs()
    print(f"Found {len(all_coeffs)} coefficients from {len(EQU)} equations.\n")

    # Build constant catalogue
    TARGETS = build_constant_catalogue()
    print(f"Built catalogue with {len(TARGETS)} mathematical constants.\n")

    # Scan for ratio relationships
    print("Scanning for ratio relationships...")
    ratio_hits = scan_ratios(all_coeffs, TARGETS, tolerance)
    total_hits = sum(len(hits) for hits in ratio_hits.values())
    print(f"Found {total_hits} ratio relationships between {len(ratio_hits)} equation pairs.\n")

    # Find integer relations
    print("Searching for integer relations...")
    integer_relations = find_integer_relations(all_coeffs, max_terms, max_coeff)
    print(f"Found {len(integer_relations)} integer relations.\n")

    # Print results
    print("\n🔍  RATIO HITS\n" + "-"*60)
    if ratio_hits:
        for (a, b), L in ratio_hits.items():
            for txt in L:
                print(f"{a:22s} vs {b:22s}: {txt}")
    else:
        print("  (none within tolerance)")

    print("\n🔑  INTEGER RELATIONS\n" + "-"*60)
    if integer_relations:
        for relation in integer_relations:
            print("  " + relation)
    else:
        print("  None found within limits.")

    # Create visualization if requested
    if create_viz:
        print("\nCreating visualization...")
        create_simple_visualization(ratio_hits)
        print("Visualization saved to 'relationships.txt'")

    return ratio_hits, integer_relations

# ============================================================
# 9.  Add custom equations (optional)
# ============================================================
def add_custom_equation(name, equation, run_analysis=True):
    """
    Add a custom equation to the library and optionally run analysis.

    Args:
        name: Name of the equation
        equation: SymPy equation
        run_analysis: Whether to run analysis after adding

    Example:
        E, m = sp.symbols('E m')
        c = sp.symbols('c', positive=True)
        add_custom_equation("Custom E=mc²", sp.Eq(E, 2*m*c**2))
    """
    EQU[name] = equation
    print(f"Added equation: {name}")

    if run_analysis:
        analyze_physics_relationships()

# ============================================================
# Run the analysis if executed directly
# ============================================================
if __name__ == "__main__":
    analyze_physics_relationships(create_viz=True)



PHYSICS CONSTANTS RELATIONSHIP ANALYSIS

Extracting coefficients from equations...
Found 11 coefficients from 20 equations.

Built catalogue with 50 mathematical constants.

Scanning for ratio relationships...
Found 37 ratio relationships between 31 equation pairs.

Searching for integer relations...
Found 1 integer relations.


🔍  RATIO HITS
------------------------------------------------------------
Coulomb law            vs Bohr radius           : 1/4/4 ≈ 1/16
Coulomb law            vs Rydberg R∞            : 1/4/1/2 ≈ 1/2
Coulomb law            vs Stefan–Boltzmann      : 1/4/1/60 ≈ 15/1
Coulomb law            vs Black-body u          : 1/4/8/15 ≈ 15/32
Coulomb law            vs Casimir P             : 1/4/-1/240 ≈ -60/1
Coulomb law            vs Golden φ              : 1/4/1/2 ≈ 1/2
Coulomb law            vs Golden φ              : 1/4/1/2 ≈ 1/2
Coulomb law            vs Schwarzschild radius  : 1/4/2 ≈ 1/8
Fine-structure α       vs Bohr radius           : 1/4/4 ≈ 1/16
Fine-struct